# setup

In [ ]:

! pip install sentinelhub[AWS]  # extra dependencies for interacting with Amazon Web Services
! pip install s2cloudless
! pip install shapely

import json
from shapely.geometry import shape # para sacar el bbox usando las coords de cada reservoir

import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentinelhub import CRS, BBox, DataCollection, MimeType, SentinelHubRequest, SHConfig, generate_evalscript, Geometry
from sentinelhub.api.catalog import get_available_timestamps
from sentinelhub.geo_utils import bbox_to_dimensions

from s2cloudless import S2PixelCloudDetector, download_bands_and_valid_data_mask

from google.colab import userdata
CLIENT_ID = userdata.get('client_id')
CLIENT_SECRET = userdata.get('client_secret')
INSTANCE_ID = userdata.get('instance_id')

config = SHConfig()

assert CLIENT_ID and CLIENT_SECRET and INSTANCE_ID #CLIENT_ID and CLIENT_SECRET and INSTANCE_ID están en secrets en el notebook
config.sh_client_id = CLIENT_ID
config.sh_client_secret = CLIENT_SECRET


# clase para almacenar la info (geometrías) particular de cada reservoir
class ReservoirShape:
  def __init__ (self, geometry, bbox, size, dataMask):
    self.geometry = geometry
    self.bbox = bbox
    self.size = size
    self.dataMask = dataMask

In [ ]:
# preparación
import pickle

def load_pickle_df(filename):
    """
    Loads a dataframe from a file saved using pickle.

    Args:
        filename (str): The name of the file to load the dataframe from.

    Returns:
        dict or None: The loaded dataframe of class instances, or None if an error occurs.
    """
    try:
        with open(filename, 'rb') as f:
            loaded_df = pickle.load(f)
        print(f"dataframe loaded from '{filename}'")
        return loaded_df
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return None
    except Exception as e:
        print(f"Error loading dataframe from '{filename}': {e}")
        return None



Mounted at /content/drive


# Útiles para datos satelitales

In [ ]:
def apiReqRez(data_collection, fecha, evalscript, reservoirShape):
  """
  Envía una solicitud a la API de Sentinel Hub y recupera datos geoespaciales.
  Args:
    data_collection (DataCollection): La colección de datos a usar (por ejemplo, DataCollection.SENTINEL2_L2A).
    fecha (str): Fecha en formato "YYYY-MM-DD" para la que tiene que existir una imagen para la misión especificada en 'data_collection'.
    evalscript (str): El Evalscript utilizado para hacer la solicitud a la API. Ver https://docs.sentinel-hub.com/api/latest/evalscript/.
  Returns:
    numpy.ndarray: La respuesta de la API. Si ha sido exitosa, un numpy.ndarray con la lectura satelital.
  """

  assert isinstance(reservoirShape, ReservoirShape), "reservoirShape param must be an instance of ReservoirShape."
  request = SentinelHubRequest(
      evalscript=evalscript,
      input_data=[
          SentinelHubRequest.input_data(
              data_collection=data_collection,
              time_interval=fecha
              )
          ],
      responses=[
          SentinelHubRequest.output_response("default", MimeType.TIFF),
      ],

      bbox=reservoirShape.bbox,
      geometry=reservoirShape.geometry,
      size=reservoirShape.size,

      config=config,
  )

  response = request.get_data() # API call

  return response[0] # el [0] porque devuelve lista de responses

DM_EVALSCRPT = '''
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["dataMask"],
                units: ["DN"]
            }],
            output: {
              bands: 1
            }
        }
    }

    function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
        outputMetadata.userData = {
            "norm_factor":  inputMetadata.normalizationFactor
        }
    }

    function evaluatePixel(sample) {
        return [sample.dataMask];
    }

  '''

NUBES_EVALSCRPT = '''
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B01", "B02", "B04", "B05", "B08", "B8A", "B09", "B10", "B11", "B12"],
                units: ["REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE"]
            }],
            output: {
              bands: 10,
              sampleType: "FLOAT32"
            }
        }
    }

    function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
        outputMetadata.userData = {
            "norm_factor":  inputMetadata.normalizationFactor
        }
    }

    function evaluatePixel(sample) {
        return [sample.B01, sample.B02, sample.B04, sample.B05, sample.B08, sample.B8A, sample.B09, sample.B10, sample.B11, sample.B12];
    }

  '''

AGUA_EVALSCRPT = '''
  //VERSION 3
    function setup() {
        return {
            input: [{
              bands: ["B03", "B08"],
              units: ["REFLECTANCE", "REFLECTANCE"]
            }],
            output: {
              bands: 1,
              sampleType: "FLOAT32"
            }

        }
    }

    //para preservar el factor de normalización
    function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
        outputMetadata.userData = {
            "norm_factor":  inputMetadata.normalizationFactor
        }
    }


    function evaluatePixel(samples) {
        return[index(samples.B03, samples.B08)];
    }


  '''

ALLBANDS_EVALSCRPT = '''
    //VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"],
                units: ["REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE"]
            }],
            output: {
              bands: 12,
              sampleType: "FLOAT32"
            }
        }
    }

    function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
        outputMetadata.userData = {
            "norm_factor":  inputMetadata.normalizationFactor
        }
    }

    function evaluatePixel(sample) {
        return [sample.B01, sample.B02, sample.B03, sample.B04, sample.B05, sample.B06, sample.B07, sample.B08, sample.B8A, sample.B09, sample.B11, sample.B12];
    }

  '''

B9x11_EVALSCRPT = '''
  //VERSION=3
  function evaluatePixel(samples) {
    let val = samples.B09*samples.B11
    return [val];
  }

  function setup() {
    return {
      input: [{
        bands: [
          "B09",
          "B11",
        ]
      }],
      output: {
        bands: 1,
        sampleType: "FLOAT32"
      }
    }
  }
'''


In [ ]:
# para conseguir las available dates hay que usar una configuration instance (q se crean en el sh dashboard) y una de sus capas, para la que qiueras comprobar las fechas
from sentinelhub.api.ogc import OgcImageService, OgcRequest
from sentinelhub.constants import ServiceType

def obtencionFechas(intervalo, reservoirShape):
  """
  Obtiene una lista de fechas para un intervalo de tiempo en las que se ha hecho una lectura
  satelital con las características de la capa 'layer4fechas' creada en el dashboard de Sentinel Hub.
  Args:
    intervalo (tuple): Una tupla que especifica el intervalo de tiempo (fecha_inicio, fecha_fin)
              en el formato ('YYYY-MM-DD', 'YYYY-MM-DD').
  Returns:
    list: Una lista ordenada de fechas únicas (cadenas en formato 'YYYY-MM-DD')
        que representan fechas válidas dentro del intervalo especificado.
  """

  ois_config = config.copy() # creamos una config nueva
  ois_config.instance_id = INSTANCE_ID # hace falta una conf. instance en la que este la layer q usamos para preguntar por las fechas
  ois_config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
  ois_config.sh_base_url = "https://sh.dataspace.copernicus.eu"
  ois = OgcImageService(config=ois_config)

  request = OgcRequest(
      layer = 'layer4fechas', # el nombre en sh dashboard de la layer con el evalscrpt = allbands_evalscript
      bbox = reservoirShape.bbox,
      data_collection = DataCollection.SENTINEL2_L2A,
      time = intervalo,
      service_type = ServiceType.WMS,
      config = ois_config,
      size_x = int(np.ceil(reservoirShape.size[0])),
      size_y = int(np.ceil(reservoirShape.size[1])),
      )
  response = ois.get_dates(request)
  fechas = [dt_obj.strftime('%Y-%m-%d') for dt_obj in response] # lista de fechas validas en formato 'yyyy-mm-dd'
  fechas = list(set(fechas)) # quitamos duplicados
  fechas.sort() # ordenamos

  return fechas

In [ ]:
aquaThres = 0.5 # se asumirán con agua los px cuyo indice en la mascara de agua > aquaThres

def imgEnFecha(fecha, minPrctgPx = .01, rezEvalscript=ALLBANDS_EVALSCRPT, reservoirShape=None):

  dataMask = reservoirShape.dataMask

  minValidPx = int(np.count_nonzero(dataMask)*minPrctgPx) # nº min de px validos del pantano
  if minValidPx == 0:
    minValidPx = 1 # cuerpos de agua que abarquen pocos px de las imagenes deberan abarcar 1 px minimo

  # Comprobacion de que hay suficientes px sin nubes
  nubesData = apiReqRez(DataCollection.SENTINEL2_L1C, fecha, NUBES_EVALSCRPT, reservoirShape)
  cloud_detector = S2PixelCloudDetector(threshold=0.4, average_over=4, dilation_size=2, all_bands=False) #OJO: Si en NUBES_EVALSCRPT no estan todas las bandas hay q cambiar param 'all_bands'
  cloudMask = cloud_detector.get_cloud_masks(nubesData[np.newaxis, ...])[0]
  cloudMasknt = ~cloudMask & dataMask # 1o: se invierte cloudmask para q zonas sin nubes = 1; 2o se aplica dataMask para poner px fuera del pantano = 0
  nPxSinNubes = int(np.count_nonzero(cloudMasknt))
  if nPxSinNubes < minValidPx:
    return None # fecha sin imagen válida

  else:
    # Comprobacion de que hay suficientes px con agua
    aguaData = apiReqRez(DataCollection.SENTINEL2_L2A, fecha, AGUA_EVALSCRPT, reservoirShape)
    aquaMask = np.where(aguaData > aquaThres, 1, 0)
    nPxSinNubesOAgua = int(np.count_nonzero(cloudMasknt & aquaMask))
    if nPxSinNubesOAgua < minValidPx:
      return None # fecha sin imagen válida
    else:
      # Se calcula y guarda el promedio de los pixeles vaidos en la imagen
      apiCallResult = apiReqRez(DataCollection.SENTINEL2_L2A, fecha, rezEvalscript, reservoirShape)
      cloudNaquaMask = aquaMask & cloudMasknt
      pxValidos = apiCallResult[cloudNaquaMask == 1] # array de los px validos (sin nubes y con agua)

      return np.mean(pxValidos) # guardamos la media de los valores px validos


# Útiles para fechas

In [ ]:
from datetime import datetime, timedelta

def find_closest_previous_date(dates_list, target_date_str):
  """
  Finds the closest previous date in a list of datetime objects.

  Args:
    dates_list: A list of datetime objects.
    target_date_str: The target date string in the format 'YYYY-MM-DD'.

  Returns:
    The closest previous date in the list in string format, or None if no such date is found.
  """

  target_date = datetime.strptime(target_date_str, '%Y-%m-%d')

  # Find closest previous date
  closest_previous_date = None
  min_diff = float('inf')
  for date in dates_list:
    diff = target_date - date
    if diff >= timedelta(0) and diff.days < min_diff:  # Check if previous and closer
      min_diff = diff.days
      closest_previous_date = date

  # Return as string if found
  return closest_previous_date.strftime('%Y-%m-%d') if closest_previous_date else None


def days_diff(date1_str, date2_str):
  """Calculates the difference in days between two dates.

  Args:
    date1_str: The first date string in 'yyyy-mm-dd' format.
    date2_str: The second date string in 'yyyy-mm-dd' format.

  Returns:
    The difference in days between the two dates, which can be 0.
  """
  date1 = datetime.strptime(date1_str, '%Y-%m-%d')
  date2 = datetime.strptime(date2_str, '%Y-%m-%d')
  diff = abs((date2 - date1).days)  # Use abs() to ensure a positive difference
  return diff


# Preparación datos valición

In [ ]:
# Se crea la forma del embalse de El Val
elVal_bbox = BBox(bbox=[-1.82727781561789, 41.8733944002713, -1.78675590600686, 41.8837564288121], crs=CRS.WGS84)
elVal_geometry = Geometry(geometry={"type":"Polygon","coordinates":[[[-1.82727781561789,41.878836997648],[-1.82706061099506,41.8793910316484],[-1.82678427776792,41.8796379207089],[-1.8267768180961,41.8800685514024],[-1.82642537447412,41.8802598648753],[-1.82617844158665,41.8806380527725],[-1.82614655470031,41.8810247360922],[-1.82543288469845,41.881535289767],[-1.82390550456733,41.8819140643813],[-1.82244400558138,41.8814295004439],[-1.82169248623425,41.8814295729659],[-1.81975201370667,41.8808426844035],[-1.81935180843586,41.8810822729263],[-1.81889142551101,41.8809638457657],[-1.81770961331756,41.8818029429609],[-1.81794192849123,41.8811437986041],[-1.81747439224727,41.8811669513082],[-1.81544009979004,41.8819473181518],[-1.81537615307333,41.8827316637033],[-1.81529526493697,41.8824714388691],[-1.81478143545625,41.8821641473958],[-1.81321515692413,41.8823354730057],[-1.81270695832156,41.8830140476162],[-1.81290528275152,41.8835253802118],[-1.81278937249098,41.8837564288121],[-1.81277806268346,41.883480857389],[-1.8122199798341,41.8828389804146],[-1.81214536293955,41.8823638613245],[-1.81120590421546,41.8824885712919],[-1.81096625650443,41.8826861040226],[-1.80966922247806,41.8821837586669],[-1.80913373180124,41.8824093077282],[-1.80903349932946,41.8828505706354],[-1.80926700515388,41.8831684218045],[-1.80887135846995,41.882985752785],[-1.80849363922767,41.8825331981089],[-1.80782983754413,41.8827851743272],[-1.80754896359537,41.8830833830551],[-1.80766280842421,41.882724078224],[-1.80816970800451,41.8821383700962],[-1.80812440206062,41.8819662306239],[-1.80697954470102,41.8815603363968],[-1.80644983192242,41.881594891384],[-1.80620458724806,41.881353327432],[-1.80553516301825,41.8811798034405],[-1.80478722669033,41.8817747810412],[-1.8047714606654,41.8821647313033],[-1.80448304441478,41.8818655050751],[-1.8045702913307,41.8809824092244],[-1.80439240437451,41.8809129497498],[-1.80338843588124,41.880930800669],[-1.80242091254617,41.8806542077364],[-1.80231133447572,41.8809671551384],[-1.80248106547165,41.8814238262763],[-1.80233655446581,41.8819055827932],[-1.80196261114899,41.8814340285499],[-1.80102987340036,41.8819104804219],[-1.80133884977586,41.8811024604306],[-1.80124510949152,41.8810009596878],[-1.80057983054835,41.8810946463983],[-1.8009589055566,41.8805299037574],[-1.80082217244091,41.880376149344],[-1.80009383695977,41.8802430838122],[-1.80001865691101,41.8801117181537],[-1.79973251752382,41.8802351537555],[-1.79982218955014,41.8803640060035],[-1.7996450567229,41.8805319458118],[-1.79890432846051,41.8808159531194],[-1.79910058217793,41.8802827647404],[-1.79903867785009,41.8796713709064],[-1.79865510127162,41.8796905758558],[-1.79851470331592,41.8795314799428],[-1.79827554450873,41.8796962972244],[-1.79826131350684,41.8799661150494],[-1.79810925650187,41.8798642175742],[-1.79806567447339,41.8801393311783],[-1.79740674119285,41.8806149905623],[-1.79758209939706,41.8799970073255],[-1.79726740311528,41.8791548765734],[-1.79666691901402,41.8793225585723],[-1.79629048646262,41.8790609927122],[-1.79588907016899,41.8790608839296],[-1.79583189437719,41.8798596999822],[-1.79527531450318,41.8806087468554],[-1.79513283759495,41.8797185158501],[-1.79427004716887,41.8801490959866],[-1.79430869413544,41.8796666299033],[-1.79388610532022,41.8796037776323],[-1.79448043157457,41.8790513270599],[-1.79418587413483,41.8786675570496],[-1.79362738343901,41.8786719576701],[-1.79338362649472,41.8783130272995],[-1.79267358583723,41.8784717482201],[-1.79243667442283,41.8778482628759],[-1.79159094158416,41.8777606361684],[-1.79101089664379,41.877448491317],[-1.79082433361377,41.8777854890534],[-1.79053541889838,41.8778298086641],[-1.79012429318485,41.8786343997579],[-1.79032926872101,41.8794731892807],[-1.79001708861323,41.8790345311159],[-1.78941471594915,41.879355086099],[-1.7896346234327,41.8786773474346],[-1.78940255422556,41.8782613000913],[-1.78878882844704,41.8783087977463],[-1.7891183673016,41.8776237950861],[-1.78817905156632,41.8777292250975],[-1.787588452025,41.8779842859756],[-1.78675590600686,41.8747519580797],[-1.78738772590401,41.8744153166891],[-1.78786772210441,41.8745495541067],[-1.78850868072584,41.8743712082576],[-1.78964644204165,41.8744307721619],[-1.78988636765576,41.8742033846321],[-1.78989056943686,41.8738597399055],[-1.79047869853603,41.873822778178],[-1.79084145191376,41.8733944002713],[-1.79127514848517,41.874646417023],[-1.79324331041029,41.8751399372827],[-1.7943505019211,41.874424719589],[-1.79433424146509,41.8754746343384],[-1.79517337956745,41.875785817354],[-1.79576978856021,41.8756535089737],[-1.79573272391819,41.8759969317443],[-1.79589112552326,41.8761806504208],[-1.79683261329047,41.8764898142374],[-1.7971950009072,41.8764076990372],[-1.7982867932155,41.8754496251635],[-1.79875103562675,41.8747763101292],[-1.79874035377039,41.8756599195633],[-1.7984170875702,41.8764433406224],[-1.79850282002217,41.8765883782049],[-1.80000762772688,41.8770922501765],[-1.80042065573986,41.8770377549775],[-1.80080329277687,41.8773949038439],[-1.80134720443857,41.8773794729995],[-1.80148628961315,41.8776477114123],[-1.80190721941164,41.8778578643629],[-1.80212642778321,41.8778374536616],[-1.80374254044806,41.8766646325469],[-1.80343895877661,41.8770280651748],[-1.80318542283659,41.8777791853363],[-1.80313159994431,41.8780161349158],[-1.80326157705146,41.8781205778189],[-1.80592253249243,41.8784003120156],[-1.80665467814352,41.8782142784239],[-1.8067069369327,41.8784191570135],[-1.80703681175234,41.8786068883906],[-1.80860897177942,41.8789801476742],[-1.80913883743092,41.8789208168677],[-1.81018506745297,41.878104172331],[-1.80982864912023,41.8792227938857],[-1.80992452066216,41.8794497560846],[-1.81184655265018,41.8798744740862],[-1.81240968416889,41.8797745507224],[-1.81253505196819,41.8799744175929],[-1.81324317252286,41.8796600263133],[-1.81370934357197,41.8791286002685],[-1.81462435427787,41.8786622906009],[-1.8143027804954,41.8789602878803],[-1.81421005048142,41.8795805519189],[-1.81700225418677,41.8792528868237],[-1.81781714061544,41.8785349516061],[-1.81797828299946,41.8788988505855],[-1.81831030230912,41.8790400190651],[-1.821061539124,41.8793032747969],[-1.8216802149523,41.8789386511012],[-1.82180207444042,41.8794476169993],[-1.82220304645525,41.8797477582006],[-1.82332933155177,41.8799276418415],[-1.82415247938706,41.879841305858],[-1.82491415612036,41.8794460256712],[-1.82727781561789,41.878836997648]]]}, crs=CRS.WGS84)
elVal_size = [512, 175.828]
elVal_dataMask = apiReqRez(DataCollection.SENTINEL2_L2A, "2024-02-22", DM_EVALSCRPT,ReservoirShape(elVal_geometry,elVal_bbox,elVal_size,None))


elVal_shape = ReservoirShape(elVal_geometry,elVal_bbox,elVal_size,elVal_dataMask)

In [ ]:
file_path = './raw_datasets/cianolecturas_elVal.csv'
try:
  datosRaw = pd.read_csv(file_path)
except FileNotFoundError:
  print(f"Error: File not found at {file_path}")
except Exception as e:
  print(f"An error occurred: {e}")

datosRaw = datosRaw[['Fecha',	'Biovolumen de cianobacteria (mm³/L)']]
datosRaw['Fecha'] = pd.to_datetime(datosRaw['Fecha'], format='%d/%m/%Y')
datosRaw = datosRaw[datosRaw['Fecha'] >= pd.Timestamp('2017-01-01')] #hay fechas previas al lanzamiento de los satelites

cianolecturas_elVal = datosRaw.rename(columns={'Fecha': 'date', 'Biovolumen de cianobacteria (mm³/L)': 'abun(mm3/L)'}) # cambia nombres columnas
cianolecturas_elVal.reset_index(drop=True, inplace=True)
cianolecturas_elVal

,date,abun(mm3/L)
0,2017-06-13,0.572
1,2017-09-12,0.072
2,2018-07-04,3.903
3,2019-07-09,0.173
4,2020-07-10,9.232
5,2020-09-10,2.613
6,2021-07-16,0.380
7,2021-09-10,0.083
8,2022-07-08,0.683
9,2022-09-20,0.030


In [ ]:
def assign_group_elVal(mm3_L):
  '''
  Para las medidas del embalse del Val las lecturas de cianobacterias están en mm3/L. Esta función
  devuelve el id del nivel de alerta según la OMS correspondiente a los mm3/L proporcionados en el argumento
  '''
  if mm3_L <= .02:
    return 0
  elif mm3_L <= .2:
    return 1
  elif mm3_L <= 2:
    return 2
  elif mm3_L <= 10:
    return 3
  elif mm3_L > 10:
    return 4
  else:
    return -1

# columnas 'promedio', 'peso' y 'etiqueta' para el entrenamiento y validación de los modelos, y
# 'og_index', que indica el índice de la cianolectura en 'cianolecturas_elVal' con la que se ha
# creado el dato final, esta última por trazabilidad en caso de que haya errores
elVal_data = pd.DataFrame(columns=['promedio','peso','label','og_index'])

for index, cianolectura in cianolecturas_elVal.iterrows():
  print('\nprocesando cianolectura ',index,'/',len(cianolecturas_elVal))
  dato = {} # dict que será el nuevo dato en elVal_data

  # primero se obtienen la lista de fechas con imágenes cercanas a la cianolectura
  doce_dias_prior = (cianolectura['date'] - timedelta(days=12)).strftime("%Y-%m-%d")
  close_dates = obtencionFechas((doce_dias_prior, cianolectura['date'].strftime("%Y-%m-%d")), elVal_shape)
  close_dates = [datetime.strptime(fecha, '%Y-%m-%d') for fecha in close_dates]

  img_date = find_closest_previous_date(close_dates, cianolectura['date'].strftime("%Y-%m-%d"))
  if img_date is None:
    print('Error: no se ha encontrado fecha con imagen en los 12 dias antes de cianolectura. (no debería ocurrir)')
    continue # no hay fecha con imagenes antes de una lectura (no debería ocurrir)

  promedio_img = imgEnFecha(img_date, rezEvalscript=B9x11_EVALSCRPT, reservoirShape=elVal_shape)
  if promedio_img is None:
    print('Imagen descartada por filtrado')
    continue # demasiadas nubes o muy poca agua
  else:
    dato['og_index'] = index
    dato['peso'] = days_diff(img_date, cianolectura['date'].strftime("%Y-%m-%d"))
    dato['promedio'] = promedio_img
    dato['label'] = assign_group_elVal(cianolectura['abun(mm3/L)'])
    elVal_data.loc[len(elVal_data)] = dato
    print('Imagen guardada:',dato)


with open('./tmp/final_data_ElVal.pkl', 'wb') as file:
  pickle.dump(elVal_data, file)


procesando cianolectura  4 / 10
Imagen guardada: {'og_index': 4, 'peso': 5, 'promedio': np.float32(0.0042035095), 'label': 2}

procesando cianolectura  5 / 10
Imagen descartada por filtrado

procesando cianolectura  6 / 10
Imagen guardada: {'og_index': 6, 'peso': 1, 'promedio': np.float32(0.0004564538), 'label': 3}

procesando cianolectura  7 / 10
Imagen guardada: {'og_index': 7, 'peso': 1, 'promedio': np.float32(0.0005361309), 'label': 1}

procesando cianolectura  8 / 10
Imagen guardada: {'og_index': 8, 'peso': 3, 'promedio': np.float32(0.001197481), 'label': 3}

procesando cianolectura  9 / 10
Imagen guardada: {'og_index': 9, 'peso': 0, 'promedio': np.float32(0.0002199586), 'label': 3}

procesando cianolectura  10 / 10
Imagen descartada por filtrado

procesando cianolectura  11 / 10
Imagen descartada por filtrado

procesando cianolectura  12 / 10
Imagen guardada: {'og_index': 12, 'peso': 1, 'promedio': np.float32(0.0013369364), 'label': 2}

procesando cianolectura  13 / 10
Imagen gu

# modelo usa datos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

# se cargan los datasets. los datos de eeuu serán los de entrenamiento y los del Val de test
raw_train_data = load_pickle_df('./tmp/final_data_Dataset4.pkl')
raw_test_data = load_pickle_df('./tmp/final_data_ElVal.pkl')
print('numero de cianolecturas de cada nivel de alerta en train_data: ')
display(raw_train_data['label'].value_counts())
print('numero de cianolecturas de cada nivel de alerta en test_data: ')
display(raw_test_data['label'].value_counts())


dataframe loaded from '/content/drive/My Drive/colaboI3A/tmp/final_data_Dataset4.pkl'
dataframe loaded from '/content/drive/My Drive/colaboI3A/tmp/final_data_ElVal.pkl'
numero de cianolecturas de cada nivel de alerta en train_data: 


,count
label,
0,296
1,68
4,64
2,18
3,12


numero de cianolecturas de cada nivel de alerta en test_data: 


,count
label,
3,3
2,2
1,2


## Separacion en atributos y etiquetas, entrenamiento y test

In [ ]:
train_data = raw_train_data.copy()
test_data = raw_test_data.copy()

# El dataset de entrenamiento esta claramente desbalanceado, se hace undersampling de la clase 0: se
# escogen 68 indices aleatorios de la clase 0 para eliminarlos y que haya tantas muestras de la
# clase 0 como de la siguiente clase mas frecuente
selected_indices = np.random.choice(np.where(train_data['label']==0)[0], size=68, replace=False)
# se crea una mascara que pone a true todos los datos que no son de la clase 0 U los que estan en selected_indices
mask = np.logical_or(train_data['label'] != 0, np.isin(np.arange(len(train_data['label'])), selected_indices))
train_data = train_data[mask]

# Separate features (X) and target (y) of train data
X_train = train_data[['promedio', 'peso']]
y_train = train_data['label']

# Separate features (X) and target (y) of test data
X_test = test_data[['promedio', 'peso']]
y_test = test_data['label']

print('x train: ',X_train.shape)
print('y train: ',y_train.shape)
print('x test: ',X_test.shape)
print('y test: ',y_test.shape)
print()
print('Train support: ')
display(y_train.value_counts())
print()
print('Test support: ')
display(y_test.value_counts())


x train:  (230, 2)
y train:  (230,)
x test:  (7, 2)
y test:  (7,)

Train support: 


,count
label,
1,68
0,68
4,64
2,18
3,12



Test support: 


,count
label,
3,3
2,2
1,2


### Atributos y etiquetas para clasificación binaria
no alertar: clases 0,1

alertar: clases 2,3,4

In [ ]:
# se cambian las etiquetas para convertirlo en un problema de clasificación binaria
train_data_bin = train_data.copy()
train_data_bin['label'] = np.where(train_data_bin['label']<2, 0, 1)
test_data_bin = test_data.copy()
test_data_bin['label'] = np.where(test_data_bin['label']<2, 0, 1)

# Separate features (X) and target (y) of train data
X_bin_train = train_data_bin[['promedio', 'peso']]
y_bin_train = train_data_bin['label']

# Separate features (X) and target (y) of test data
X_bin_test = test_data_bin[['promedio', 'peso']]
y_bin_test = test_data_bin['label']

print('x_bin train: ',X_bin_train.shape)
print('y_bin train: ',y_bin_train.shape)
print('x_bin test: ',X_bin_test.shape)
print('y_bin test: ',y_bin_test.shape)
print()
print('Train support: ')
display(y_bin_train.value_counts())
print()
print('Test support: ')
display(y_bin_test.value_counts())

x_bin train:  (230, 2)
y_bin train:  (230,)
x_bin test:  (7, 2)
y_bin test:  (7,)

Train support: 


,count
label,
0,136
1,94



Test support: 


,count
label,
1,5
0,2


## funciones genéricas

In [ ]:
def evaluacion_generica(modelo, modelo_bin):
  """
  Esta funcion entrena y evalúa dos modelos: uno para clasificación multiclase y otro para
  clasificación binaria.

  Args:
    modelo: Modelo para clasificación multiclase.
    modelo_bin: Modelo para clasificación binaria.

  Returns:
    None. Imprime las métricas de evaluación para ambos modelos.
  """

  # Entrenamiento
  modelo.fit(X_train, y_train)
  modelo_bin.fit(X_bin_train, y_bin_train)

  # Predicciones
  y_pred = modelo.predict(X_test)
  y_pred_bin = modelo_bin.predict(X_bin_test)

  # Accuracy, classification report y matriz de confusión
  print('-'*20+' Clasificación multiclase '+'-'*20)
  print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
  print(classification_report(y_test, y_pred,zero_division=0))
  print(confusion_matrix(y_test, y_pred))
  print()
  print('-'*20+' Clasificación binaria '+'-'*20)
  print(f"Accuracy: {accuracy_score(y_bin_test, y_pred_bin)}")
  print(classification_report(y_bin_test, y_pred_bin,zero_division=0))
  print(confusion_matrix(y_bin_test, y_pred_bin))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

def grid_search_y_evaluacion(modelo, param_grid):
  """
  Realiza una búsqueda de cuadrícula (grid search) para encontrar los mejores hiperparámetros de un
  modelo para clasificacion *binaria* utilizando validación cruzada estratificada y evalúa el modelo
  con los mejores hiperparámetros encontrados con los datos de prueba del embalse del Val.

  Args:
    modelo: El modelo a optimizar.
    param_grid: Un diccionario que define la cuadrícula de hiperparámetros a explorar.

  Returns:
    None. Imprime los mejores hiperparámetros, la precisión en validación cruzada y las métricas
    de evaluación con los datos de prueba.
  """
  # Crea el objeto GridSearchCV usando stratifiedkfold para disminuir el efecto del desbalanceado
  # de los conjuntos de entrenamiento y test
  grid_search = GridSearchCV(modelo, param_grid, cv=StratifiedKFold(n_splits=5), scoring='accuracy', n_jobs=-1)

  # Ajusta el modelo a los datos de entrenamiento
  grid_search.fit(X_bin_train, y_bin_train)

  # Imprime los mejores hiperparámetros y la mejor puntuación
  print(f"Mejores hiperparámetros: {grid_search.best_params_}")
  print(f"Mejor accuracy en validación cruzada: {grid_search.best_score_}")

  print()
  print('Prueba con datos de test (embalse del Val)')
  # Evalúa el modelo con los mejores hiperparámetros con los datos de test
  best_model_found = grid_search.best_estimator_
  y_pred = best_model_found.predict(X_bin_test)

  # Imprime las métricas de evaluación
  print(f"Accuracy para el Val: {accuracy_score(y_bin_test, y_pred)}")
  print(classification_report(y_bin_test, y_pred,zero_division=0))
  print(confusion_matrix(y_bin_test, y_pred))

## Regresión Logística

In [ ]:
from sklearn.linear_model import LogisticRegression
# Modelo reg. logística básico
logReg = LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=34)
logReg_bin = LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=34)

# se evalua este primer intento de modelos
evaluacion_generica(logReg, logReg_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.14285714285714285
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         2
           3       0.50      0.33      0.40         3

    accuracy                           0.14         7
   macro avg       0.12      0.08      0.10         7
weighted avg       0.21      0.14      0.17         7

[[0 0 0 0]
 [2 0 0 0]
 [1 0 0 1]
 [2 0 0 1]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.5714285714285714
              precision    recall  f1-score   support

           0       0.40      1.00      0.57         2
           1       1.00      0.40      0.57         5

    accuracy                           0.57         7
   macro avg       0.70      0.70      0.57         7
weighted avg       0.83      0.57      0.57         7

[[

### Optimización hiperparámetros - pruebas

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

# Define la malla de hiperparámetros a explorar
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.1, 1, 10, 100],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 1000, 2000]
}

# Crea el modelo de regresión logística
logReg = LogisticRegression(class_weight='balanced', random_state=34)

# Crea el objeto GridSearchCV usando stratifiedkfold para disminuir el efecto del desbalanceado
# de los conjuntos de entrenamiento y test
grid_search = GridSearchCV(logReg, param_grid, cv=StratifiedKFold(n_splits=5), scoring='accuracy', n_jobs=-1)

# Ajusta el modelo a los datos de entrenamiento
grid_search.fit(X_bin_train, y_bin_train)

# Imprime los mejores hiperparámetros y la mejor puntuación
print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor accuracy en validación cruzada: {grid_search.best_score_}")

print()
print('Prueba con datos de test (embalse del Val)')
# Evalúa el modelo con los mejores hiperparámetros con los datos de test
best_logReg_grid = grid_search.best_estimator_
y_pred = best_logReg_grid.predict(X_bin_test)

# Imprime las métricas de evaluación
print(f"Accuracy para el Val: {accuracy_score(y_bin_test, y_pred)}")
print(classification_report(y_bin_test, y_pred,zero_division=0))
print(confusion_matrix(y_bin_test, y_pred))

Mejores hiperparámetros: {'C': 0.1, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Mejor accuracy en validación cruzada: 0.5478260869565218

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.2857142857142857
              precision    recall  f1-score   support

           0       0.29      1.00      0.44         2
           1       0.00      0.00      0.00         5

    accuracy                           0.29         7
   macro avg       0.14      0.50      0.22         7
weighted avg       0.08      0.29      0.13         7

[[2 0]
 [5 0]]


In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# Define la función objetivo a minimizar
def objective(params):
    model = LogisticRegression(**params, class_weight='balanced')
    model.fit(X_train, y_train)
    score = accuracy_score(y_test, model.predict(X_test))
    return {'loss': -score, 'status': STATUS_OK}

# Define el espacio de búsqueda de hiperparámetros
space = {
    'penalty': hp.choice('penalty', ['l1', 'l2']),
    'C': hp.loguniform('C', -1, 1),
    'solver': hp.choice('solver', ['liblinear', 'saga']),
    'max_iter': hp.choice('max_iter', [1000, 2000])
}

# Ejecuta la optimización
trials = Trials()
best_params = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)

# Imprime los mejores hiperparámetros
print(f"Mejores hiperparámetros: {best_params}")

# Convert 'penalty' and 'solver' values to strings
penalty_mapping = {0: 'l1', 1: 'l2'}
solver_mapping = {0: 'liblinear', 1: 'saga'}

best_params['penalty'] = penalty_mapping[best_params['penalty']]
best_params['solver'] = solver_mapping[best_params['solver']]

# Crea un modelo con los mejores hiperparámetros
best_model = LogisticRegression(**best_params, class_weight='balanced').fit(X_bin_train, y_bin_train)
y_pred = best_model.predict(X_bin_test)
best_score = accuracy_score(y_bin_test, y_pred)

print(f"Precisión con los mejores hiperparámetros: {best_score}")


100%|██████████| 100/100 [00:01<00:00, 91.29trial/s, best loss: -0.14285714285714285]
Mejores hiperparámetros: {'C': np.float64(0.6853584066504389), 'max_iter': np.int64(1), 'penalty': np.int64(1), 'solver': np.int64(1)}
Precisión con los mejores hiperparámetros: 0.2857142857142857


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:

# hay que instalar optuna
!pip install optuna

In [ ]:
import optuna

# Define la función objetivo a maximizar
def objective(trial):
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    C = trial.suggest_float('C', 0.1, 10, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'saga'])
    max_iter = trial.suggest_categorical('max_iter', [1000, 2000])

    model = LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=max_iter, class_weight='balanced')
    model.fit(X_train, y_train)
    score = accuracy_score(y_test, model.predict(X_test))
    return score

# Crea el estudio de Optuna
study = optuna.create_study(direction='maximize')  # Maximizar accuracy

# Ejecuta la optimización
study.optimize(objective, n_trials=100)  # Ajusta n_trials

# Imprime los mejores hiperparámetros y la mejor puntuación
best_params = study.best_params

print('\nResultados de la búsqueda:')
print(f"Mejores hiperparámetros: {best_params}")
print(f"Mejor puntuación: {study.best_value}")

[I 2025-05-15 11:51:20,392] A new study created in memory with name: no-name-98b45517-5da5-4ad4-8226-c6bf7d051f54
[I 2025-05-15 11:51:20,400] Trial 0 finished with value: 0.0 and parameters: {'penalty': 'l1', 'C': 0.1451319433665421, 'solver': 'liblinear', 'max_iter': 1000}. Best is trial 0 with value: 0.0.
[I 2025-05-15 11:51:20,461] Trial 1 finished with value: 0.14285714285714285 and parameters: {'penalty': 'l2', 'C': 8.573889164975613, 'solver': 'saga', 'max_iter': 2000}. Best is trial 1 with value: 0.14285714285714285.
[I 2025-05-15 11:51:20,469] Trial 2 finished with value: 0.14285714285714285 and parameters: {'penalty': 'l2', 'C': 0.2538369097681049, 'solver': 'saga', 'max_iter': 1000}. Best is trial 1 with value: 0.14285714285714285.
[I 2025-05-15 11:51:20,475] Trial 3 finished with value: 0.0 and parameters: {'penalty': 'l2', 'C': 1.4095952511452563, 'solver': 'liblinear', 'max_iter': 1000}. Best is trial 1 with value: 0.14285714285714285.
[I 2025-05-15 11:51:20,482] Trial 4 f


Resultados de la búsqueda:
Mejores hiperparámetros: {'penalty': 'l2', 'C': 8.573889164975613, 'solver': 'saga', 'max_iter': 2000}
Mejor puntuación: 0.14285714285714285


## Random forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Modelo rand. forest
randForest = RandomForestClassifier(n_estimators=200, random_state=34, class_weight='balanced_subsample', n_jobs=-1)
randForest_bin = RandomForestClassifier(n_estimators=200, random_state=34, class_weight='balanced_subsample', n_jobs=-1)

# se evalúan este primer intento de modelos
evaluacion_generica(randForest, randForest_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.14285714285714285
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         2
           3       0.50      0.33      0.40         3
           4       0.00      0.00      0.00         0

    accuracy                           0.14         7
   macro avg       0.10      0.07      0.08         7
weighted avg       0.21      0.14      0.17         7

[[0 0 0 0 0]
 [0 0 0 1 1]
 [0 0 0 0 2]
 [1 0 0 1 1]
 [0 0 0 0 0]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.5714285714285714
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.67      0.80      0.73         5

    accuracy                           0.57         7
   macro avg       0.33      0.40   

### Intento de mejora de hiperparámetros para la **clasificación binaria**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Define la malla de hiperparámetros a explorar
param_grid = {
    'n_estimators': [100, 200, 300],  # Número de árboles en el bosque
    'max_depth': [None, 5, 10],  # Profundidad máxima de cada árbol
    'min_samples_split': [2, 5, 10],  # Número mínimo de muestras para dividir un nodo
    'min_samples_leaf': [1, 2, 4],  # Número mínimo de muestras en un nodo hoja
}

# Crea el modelo de Bosque Aleatorio
randForest = RandomForestClassifier(class_weight='balanced', random_state=34, n_jobs=-1)  # class_weight='balanced' para datos desbalanceados

grid_search_y_evaluacion(randForest, param_grid)

Mejores hiperparámetros: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Mejor accuracy en validación cruzada: 0.726086956521739

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.5714285714285714
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.67      0.80      0.73         5

    accuracy                           0.57         7
   macro avg       0.33      0.40      0.36         7
weighted avg       0.48      0.57      0.52         7

[[0 2]
 [1 4]]


## GradientBoost

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
# Modelos GradientBoost para clasificación multiclase y binaria
gradBoost = GradientBoostingClassifier(random_state=34, n_estimators=200)
gradBoost_bin = GradientBoostingClassifier(random_state=34, n_estimators=200)

# se evalua este primer intento de modelos
evaluacion_generica(gradBoost, gradBoost_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.14285714285714285
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         2
           3       1.00      0.33      0.50         3
           4       0.00      0.00      0.00         0

    accuracy                           0.14         7
   macro avg       0.20      0.07      0.10         7
weighted avg       0.43      0.14      0.21         7

[[0 0 0 0 0]
 [1 0 0 0 1]
 [1 0 0 0 1]
 [1 0 1 1 0]
 [0 0 0 0 0]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65   

### Intento de mejora de hiperparámetros para la **clasificación binaria**:

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Define la malla de hiperparámetros a explorar
param_grid = {
    'n_estimators': [100, 200, 300],  # Número de etapas de boosting
    'learning_rate': [0.01, 0.1, 0.2],  # Tasa de aprendizaje (reduce el tamaño del paso para prevenir sobreajuste)
    'max_depth': [3, 5, 7],  # Profundidad máxima de los estimadores de regresión individuales
    'min_samples_split': [2, 5, 10],  # Número mínimo de muestras para dividir un nodo interno
    'min_samples_leaf': [1, 2, 4],  # Número mínimo de muestras requeridas para ser un nodo hoja
}

# Se crea el modelo Gradient Boosting
gradBoost = GradientBoostingClassifier(random_state=34)

grid_search_y_evaluacion(gradBoost, param_grid)

Mejores hiperparámetros: {'learning_rate': 0.2, 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 300}
Mejor accuracy en validación cruzada: 0.7260869565217392

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65      0.65         7
weighted avg       0.71      0.71      0.71         7

[[1 1]
 [1 4]]


## AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

adaBoost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2, random_state=34),
                             n_estimators=200, random_state=34)

adaBoost_bin = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2, random_state=34),
                             n_estimators=200, random_state=34)

evaluacion_generica(adaBoost, adaBoost_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.14285714285714285
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.50      0.67         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00         0

    accuracy                           0.14         7
   macro avg       0.20      0.10      0.13         7
weighted avg       0.29      0.14      0.19         7

[[0 0 0 0 0]
 [0 1 0 0 1]
 [0 0 0 0 2]
 [1 0 0 0 2]
 [0 0 0 0 0]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65   

### Intento de mejora de hiperparámetros para la **clasificación binaria**:

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200, 300],  # Número de estimadores (árboles débiles)
    'learning_rate': [0.01, 0.1, 1, 1.5],  # Tasa de aprendizaje
    'estimator__max_depth': [1, 2, 3, 4] # Profundidad máxima del estimador base (DecisionTree)
}

adaBoost = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=34), random_state=34)
grid_search_y_evaluacion(adaBoost, param_grid)

Mejores hiperparámetros: {'estimator__max_depth': 3, 'learning_rate': 1, 'n_estimators': 100}
Mejor accuracy en validación cruzada: 0.7043478260869565

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65      0.65         7
weighted avg       0.71      0.71      0.71         7

[[1 1]
 [1 4]]


## XGBoost

In [ ]:
import xgboost as xgb

xgBoost = xgb.XGBClassifier(objective='multi:softmax', num_class=5, seed=34)
xgBoost_bin = xgb.XGBClassifier(objective='binary:logistic', seed=34)

evaluacion_generica(xgBoost, xgBoost_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.14285714285714285
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.50      0.67         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00         0

    accuracy                           0.14         7
   macro avg       0.20      0.10      0.13         7
weighted avg       0.29      0.14      0.19         7

[[0 0 0 0 0]
 [0 1 0 1 0]
 [0 0 0 0 2]
 [1 0 0 0 2]
 [0 0 0 0 0]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65   

### Intento de mejora de hiperparámetros para la **clasificación binaria**:

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],  # Number of trees
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # Step size shrinkage
    'max_depth': [3, 5, 7, 9],             # Maximum depth of a tree
    'gamma': [0, 0.1, 0.2, 0.5],           # Minimum gain (loss reductioin) required to make a further partition
    'reg_lambda': [0.1, 1.0, 10.0]         # L2 regularization term on weights
}


xgBoost = xgb.XGBClassifier(objective='binary:logistic', seed=34)
grid_search_y_evaluacion(xgBoost, param_grid)

Mejores hiperparámetros: {'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 500, 'reg_lambda': 1.0}
Mejor accuracy en validación cruzada: 0.7217391304347827

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65      0.65         7
weighted avg       0.71      0.71      0.71         7

[[1 1]
 [1 4]]


## SVM

In [ ]:
from sklearn.svm import SVC
svm_model = SVC()
svm_model_bin = SVC()

evaluacion_generica(svm_model, svm_model_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       2.0
           2       0.00      0.00      0.00       2.0
           3       0.00      0.00      0.00       3.0
           4       0.00      0.00      0.00       0.0

    accuracy                           0.00       7.0
   macro avg       0.00      0.00      0.00       7.0
weighted avg       0.00      0.00      0.00       7.0

[[0 0 0 0 0]
 [1 0 0 0 1]
 [1 1 0 0 0]
 [1 1 0 0 1]
 [0 0 0 0 0]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.2857142857142857
              precision    recall  f1-score   support

           0       0.29      1.00      0.44         2
           1       0.00      0.00      0.00         5

    accuracy                           0.29         7
   macro avg       0.14      0.50      0.22         

### Intento de mejora de hiperparámetros para la **clasificación binaria**:

In [ ]:
param_grid_svm = {
    'C': [0.1, 1, 10, 100],  # Parámetro de regularización
    'kernel': ['linear', 'rbf', 'poly'],  # Tipo de kernel a utilizar
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],  # Coeficiente del kernel para 'rbf' y 'poly'
    'degree': [2, 3, 4]  # Grado para el kernel 'poly' (ignorado por otros kernels)
}

svm_model = SVC()
grid_search_y_evaluacion(svm_model, param_grid_svm)

Mejores hiperparámetros: {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'linear'}
Mejor accuracy en validación cruzada: 0.591304347826087

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.2857142857142857
              precision    recall  f1-score   support

           0       0.29      1.00      0.44         2
           1       0.00      0.00      0.00         5

    accuracy                           0.29         7
   macro avg       0.14      0.50      0.22         7
weighted avg       0.08      0.29      0.13         7

[[2 0]
 [5 0]]


## LightGBM

In [ ]:
import lightgbm as lgb

num_classes = len(np.unique(y_train))
lightGBM = lgb.LGBMClassifier(objective='multiclass', num_class=num_classes, random_state=42, verbose=-1)
lightGBM_bin = lgb.LGBMClassifier(objective='binary', random_state=42, verbose=-1)
evaluacion_generica(lightGBM,lightGBM_bin)

-------------------- Clasificación multiclase --------------------
Accuracy: 0.2857142857142857
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.50      0.67         2
           2       0.00      0.00      0.00         2
           3       1.00      0.33      0.50         3
           4       0.00      0.00      0.00         0

    accuracy                           0.29         7
   macro avg       0.40      0.17      0.23         7
weighted avg       0.71      0.29      0.40         7

[[0 0 0 0 0]
 [0 1 0 0 1]
 [0 0 0 0 2]
 [1 0 0 1 1]
 [0 0 0 0 0]]

-------------------- Clasificación binaria --------------------
Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65    

### Intento de mejora de hiperparámetros para la **clasificación binaria**:

In [ ]:
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 64, 128],  # Controla la complejidad del árbol
    'subsample': [0.8, 0.9, 1.0], # Fracción de muestras para entrenar cada árbol (bagging)
    'colsample_bytree': [0.8, 0.9, 1.0] # Fracción de features para entrenar cada árbol
}

lightGBM = lgb.LGBMClassifier(objective='binary', random_state=42, verbose=-1)
grid_search_y_evaluacion(lightGBM, param_grid)

Mejores hiperparámetros: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'num_leaves': 31, 'subsample': 0.8}
Mejor accuracy en validación cruzada: 0.6652173913043478

Prueba con datos de test (embalse del Val)
Accuracy para el Val: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.80      0.80      0.80         5

    accuracy                           0.71         7
   macro avg       0.65      0.65      0.65         7
weighted avg       0.71      0.71      0.71         7

[[1 1]
 [1 4]]


## Red neuronal sencilla

Se crea una red neuronal sencilla (no más de 3 capas ocultas o más de 128 neuronas por capa) y se optimizan sus hiperparámetros.

In [ ]:

!pip install tensorflow
!pip install keras-tuner

In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
import numpy as np
import shutil # para eliminar el directorio que crea keras durante la optimización

# Para crear una RN y optimziar sus hiperparámetros se usará la librería keras_tuner

# El 1er paso es crear una clase que herede de 'HyperModel', en la que se define el esqueleto de
# la RN y, dentro de dicho esqueleto, los hiperparámetros que se desea optimizar
class BinaryClassifierHyperModel(kt.HyperModel):
    def __init__(self, input_shape):
        self.input_shape = input_shape # el tamaño de la capa de entrada (nº de atributos)

    def build(self, hp): # en este método se define el esqueleto de la RN
        """Construye un modelo Keras con hiperparámetros ajustables."""
        model = keras.Sequential() #
        model.add(keras.layers.Input(shape=self.input_shape)) # capa de entrada (input)

        # Se ajusta el número de capas ocultas
        # El número de capas ocultas es un hiperparámetro a optimizar, y con el framework elegido
        # se usa hp.Int('num_layers',1,3) para que la optimización encuentre el número adecuado
        for i in range(hp.Int('num_layers', 1, 3)):
            model.add(keras.layers.Dense(
                units=hp.Int(f'units_{i}', min_value=16, max_value=128, step=16), # nº neuronas
                activation=hp.Choice(f'activation_{i}', ['relu', 'tanh']) # fºn de activación de cada capa oculta
            ))
            model.add(keras.layers.Dropout(
                # Se añade una capa de dropout para evitar sobreajuste. El porcentaje de neuronas
                # que se desactivan será un hp entre .1 y .5
                rate=hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)
            ))

        # Capa de salida, con función sigmoide (las clasificaciones serán probabilidades en [0,1])
        model.add(keras.layers.Dense(1, activation='sigmoid'))

        # El learning rate también es un hiperparámetro que optimizar, y los valores posibles
        # estarán en [10^-4,10^-2] dando pasos logarítmicos (sampling="log")
        learning_rate = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        return model

# Una vez creada la clase que representará el modelo a optimizar se procede a la optimización
def optimize_binary_classifier(X_train, y_train):
    """
    Realiza el ajuste de hiperparámetros y borra el directorio que se crea en el proceso
    """
    input_shape = (X_train.shape[1],)
    hypermodel = BinaryClassifierHyperModel(input_shape=input_shape)

    directory_name = 'temp_keras_tuner_dir'

    # el objeto RandomSearch definirá una búsqueda aleatoria de los mejores hiperparámetros
    tuner = kt.RandomSearch(
        hypermodel, # el modelo antes definido a optimizar
        objective='val_accuracy', # la métrica que maximizará la búsqueda será la precisión
        max_trials=25, # total de combinaciones distintas de hiperparámetros que probar
        executions_per_trial=2, # con el mismo cjto de hp se entrena y valida 2 veces, para reducir el impacto de la aleatoreidad
        directory=directory_name,
        project_name='binary_classification_tuning',
        overwrite=True
    )

    stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

    print("\nComenzando optimización de hiperparámetros...")
    tuner.search(
        X_train,
        y_train,
        epochs=50, # nº máx de epochs
        validation_split=0.2,
        callbacks=[stop_early], # se aplica early stopping para no ejecutar epochs innecesarias sin mejoras sustanciales
        verbose=1
    )

    print("\nOptimización completa. Mejores hiperparámetros:")
    # Corrected call without 'num_models'
    best_hps = tuner.get_best_hyperparameters()[0]

    print(f"""
    - Número de capas: {best_hps.get('num_layers')}
    - Learning Rate: {best_hps.get('lr'):.4f}
    """)
    for i in range(best_hps.get('num_layers')):
          print(f"  - Capa {i+1} neuronas: {best_hps.get(f'units_{i}')}")
          print(f"  - Capa {i+1} funcion act.: {best_hps.get(f'activation_{i}')}")
          print(f"  - Capa {i+1} dropout: {best_hps.get(f'dropout_{i}'):.2f}")

    best_model = tuner.get_best_models(num_models=1)[0]

    print("\nResumen del mejor modelo encontrado:")
    best_model.summary()

    # Se limpia el directorio temporal usado por keras
    print(f"\nBorrando el directorio temporal..: {directory_name}")
    shutil.rmtree(directory_name)

    return best_model


# Run the optimization
best_nn_model = optimize_binary_classifier(X_bin_train, y_bin_train)

print("\nOptimización completada.")


Trial 25 Complete [00h 00m 12s]
val_accuracy: 0.54347825050354

Best val_accuracy So Far: 0.6304348111152649
Total elapsed time: 00h 05m 40s

Optimización completa. Mejores hiperparámetros:

    - Número de capas: 2
    - Learning Rate: 0.0092
    
  - Capa 1 neuronas: 80
  - Capa 1 funcion act.: relu
  - Capa 1 dropout: 0.30
  - Capa 2 neuronas: 128
  - Capa 2 funcion act.: tanh
  - Capa 2 dropout: 0.10

Resumen del mejor modelo encontrado:


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 80)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,737 (41.94 KB)

 Trainable params: 10,737 (41.94 KB)

 Non-trainable params: 0 (0.00 B)


Borrando el directorio temporal..: temp_keras_tuner_dir

Optimización completada.


In [ ]:
# Se evalua la mejor RN encontrada
best_nn_model.fit(X_bin_train, y_bin_train)
# Predicciones
y_pred_bin_proba = best_nn_model.predict(X_bin_test)
y_pred_bin = (y_pred_bin_proba >= 0.5).astype(int)
# Accuracy, classification report y matriz de confusión
print('-'*20+' Clasificación binaria '+'-'*20)
print(f"Accuracy: {accuracy_score(y_bin_test, y_pred_bin)}")
print(classification_report(y_bin_test, y_pred_bin,zero_division=0))
print(confusion_matrix(y_bin_test, y_pred_bin))


8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5562 - loss: 0.6925
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
-------------------- Clasificación binaria --------------------
Accuracy: 0.2857142857142857
              precision    recall  f1-score   support

           0       0.29      1.00      0.44         2
           1       0.00      0.00      0.00         5

    accuracy                           0.29         7
   macro avg       0.14      0.50      0.22         7
weighted avg       0.08      0.29      0.13         7

[[2 0]
 [5 0]]


Parece que, pese haber tomado medidas para evitarlo con las capas de dropout, la red neuronal se ha sobreajustado y clasifica toda muestra como de la clase 0 porque hay más muestras de la clase 0 en el conjunto de entrenamiento